# Problem 3: Fine-tune a DistilBERT model

This starter code is organized so that all of the **data engineering** (downloading the dataset, splitting it, tokenizing it, wrapping it in `Dataset`/`DataLoader` objects) and the **generic training/evaluation loop** are already written for you and fully working -- you should not need to change any of that code. Cells marked **(given)** need no edits.

Your actual tasks (Tasks 1-7 plus the bonus) are about the modeling questions: what does freezing the backbone actually buy you, how do you tune a hyperparameter properly, and what is the model's attention actually doing. Each task below has a clearly marked `# BEGIN YOUR CODE HERE` / `# END YOUR CODE HERE` block -- that is the *only* place you need to write code for that task.

In [13]:
# If you are running this notebook on Google Colab, uncomment the two
# lines below to connect to your Google Drive (e.g. to save model
# checkpoints so they survive after your Colab runtime disconnects).
# Nothing else in this notebook requires Drive -- the dataset is fetched
# directly from a public URL below -- so this cell is optional.

# from google.colab import drive
# drive.mount('/content/drive/')

## Setup (given)

In [14]:
import time
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, DistilBertForSequenceClassification

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')  # Apple Silicon GPU
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

# On MPS, the fused/SDPA attention kernel doesn't support dropout and
# raises `NotImplementedError: scaled_dot_product_attention for MPS does
# not support dropout` the instant you call .backward() in training mode.
# ATTN_IMPL picks a numerically-equivalent fallback on MPS; pass
# `attn_implementation=ATTN_IMPL` to every DistilBertForSequenceClassification
# .from_pretrained(...) call you write below (Tasks 3, 4, 5, and the Bonus),
# not just the one already given here.
ATTN_IMPL = 'eager' if device.type == 'mps' else 'sdpa'

Using device: mps


## Load and split the data (given)

The dataset is a set of ~10k COVID-era tweets, each labeled with whether it makes a medical claim (`label=1`) or not (`label=0`) -- this is the binary classification task described in the assignment. Note the dataset is fairly imbalanced (roughly 87% class 1, 13% class 0); keep that in mind when you look at raw accuracy vs. F1-macro later on.

In [15]:
DATA_URL = 'https://raw.githubusercontent.com/LCS2-IIITD/LESA-EACL-2021/main/data/Twitter.csv'

df = pd.read_csv(DATA_URL)
df = df.rename(columns={'tweet_text': 'text', 'claim': 'label'})
df = df.dropna(subset=['text', 'label']).reset_index(drop=True)
df['label'] = df['label'].astype(int)

print(f'Total examples: {len(df)}')
print(df['label'].value_counts(normalize=True))

# Stratified train/val/test split (70/15/15).
train_df, temp_df = train_test_split(
    df, test_size=0.3, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

print(f'train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}')

Total examples: 9981
label
1    0.87386
0    0.12614
Name: proportion, dtype: float64
train: 6986, val: 1497, test: 1498


## Tokenize and wrap in DataLoaders (given)

In [16]:
MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 128
BATCH_SIZE = 32

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        encodings = tokenizer(
            list(texts), padding='max_length', truncation=True,
            max_length=max_length, return_tensors='pt')
        self.input_ids = encodings['input_ids']
        self.attention_mask = encodings['attention_mask']
        self.labels = torch.tensor(labels.values, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx],
        }


train_dataset = TweetDataset(train_df['text'], train_df['label'], tokenizer, MAX_LENGTH)
val_dataset = TweetDataset(val_df['text'], val_df['label'], tokenizer, MAX_LENGTH)
test_dataset = TweetDataset(test_df['text'], test_df['label'], tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Task 4 asks you to do a final training run on train+val combined.
trainval_dataset = ConcatDataset([train_dataset, val_dataset])
trainval_loader = DataLoader(trainval_dataset, batch_size=BATCH_SIZE, shuffle=True)

## Generic training/evaluation loop (given)

These three functions are reused, unchanged, by every task below. `train_for_epochs` prints validation F1-macro after every epoch so you can watch training progress; it works regardless of whether the backbone is frozen or not, since that's controlled entirely by which parameters you hand to the optimizer and by each parameter's own `requires_grad` flag -- not by anything in these functions.

In [17]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    """Returns (f1_macro, all_preds, all_labels) over an entire loader."""
    model.eval()
    all_preds, all_labels = [], []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        preds = outputs.logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(batch['labels'].cpu().tolist())
    return f1_score(all_labels, all_preds, average='macro'), all_preds, all_labels


def train_for_epochs(model, train_loader, val_loader, optimizer, num_epochs):
    """Trains for num_epochs, printing train loss + val F1-macro each epoch."""
    for epoch in range(num_epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer)
        val_f1, _, _ = evaluate(model, val_loader)
        print(f'  epoch {epoch + 1}/{num_epochs} -- train loss: {train_loss:.4f}, val F1-macro: {val_f1:.4f}')
    return model


GRID_EPOCHS = 2   # epochs per grid-search setting (Task 3) -- kept short since you'll re-run this per setting
FINAL_EPOCHS = 3  # epochs for the final train+val runs (Tasks 4 and 5)

## Load the pretrained model (given)

**You will see a load report/warning listing some `UNEXPECTED` and `MISSING` keys -- this is expected, not an error:** `distilbert-base-uncased` is a checkpoint pretrained as a masked-language model, so it has MLM-specific weights (`vocab_transform`, `vocab_layer_norm`, `vocab_projector`) that don't exist in `DistilBertForSequenceClassification` and are safely dropped (`UNEXPECTED`); conversely, the classification head (`pre_classifier`, `classifier`) doesn't exist in the base checkpoint at all, so it is randomly initialized (`MISSING`). That randomly-initialized head is exactly what you'll be training in the tasks below.

In [18]:
model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, attn_implementation=ATTN_IMPL)
model.to(device)
print(model)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


## Task 1 (3 points): Tensor shapes through your own model

The cell below (given) prints the configuration values you need -- note `model.config.dim` is the *combined* hidden size (before splitting into heads); divide by `n_heads` yourself to get the per-head size. These shapes come from the model's fixed architecture, not from training, so you can do this before any of the tasks below. Using `dim`, `n_heads`, `n_layers`, the batch size `B` (`BATCH_SIZE`), and `max_length` `L` (`MAX_LENGTH`), compute -- showing your work -- the six quantities (a)-(f) listed in the assignment (shapes of Q/K/V per-head and combined, the attention map, Dropout-1's output, the final Output, and the self-attention sub-layer's total parameter count as a formula and as a number).

In [19]:
# Note: model.config.dim is the COMBINED hidden size (i.e., before
# splitting into heads) -- Q/K/V are each projected to this full size
# first; only afterward is that split into n_heads heads of size
# dim // n_heads each.
print(f'dim (combined hidden size, before splitting into heads): {model.config.dim}')
print(f'n_heads: {model.config.n_heads}')
print(f'n_layers: {model.config.n_layers}')
print(f'batch size B used for training: {BATCH_SIZE}')
print(f'max_length L used for training: {MAX_LENGTH}')

dim (combined hidden size, before splitting into heads): 768
n_heads: 12
n_layers: 6
batch size B used for training: 32
max_length L used for training: 128


### Task 1 answer

_Fill in each shape/quantity below, showing your work. Double-click this cell to edit._

(a) Shape of Q, K, V (single head): ...

(b) Shape of Q, K, V (combined): ...

(c) Shape of the attention map: ...

(d) Shape of Dropout-1's output: ...

(e) Shape of Output: ...

(f) Total parameters in one self-attention sub-layer (formula, then a number for your model): ...

## Task 2 (3 points): Before you freeze anything

Inspect the model's structure via `model.named_parameters()` (or `print(model)`), then fill in the blanks:

(a) Compute the total number of parameters in the model, and separately the number of parameters belonging only to the classification head (`pre_classifier` and `classifier`) -- not the DistilBERT backbone.
(b) Based on these two counts, what fraction of the model's parameters will actually receive gradient updates if you freeze the backbone?
(c) In 1-2 sentences, predict how this should affect training speed and memory use compared to fine-tuning the whole model -- you will check this prediction in Task 5.

In [20]:
#Inspect the model's structure via `model.named_parameters()` (or `print(model)`)
print(model)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [21]:
# BEGIN YOUR CODE HERE (~2 lines)
# (a) Compute the total number of parameters in `model`, and separately
#     the number of parameters in ONLY the classification head
#     (model.pre_classifier and model.classifier).
# Hint: a parameter tensor `p` has `p.numel()` elements; `model.named_parameters()`
# gives you (name, tensor) pairs, so you can tell head params apart from
# backbone params by whether the name starts with 'pre_classifier'/'classifier'.
total_params =sum(p.numel() for p in model.parameters())
head_params =sum(p.numel() for name, p in model.named_parameters() if name.startswith('pre_classifier') or name.startswith('classifier'))
# END YOUR CODE HERE

print(f'Total parameters: {total_params:,}')
print(f'Classification head parameters: {head_params:,}')
print(f'Fraction trainable if backbone is frozen: {head_params / total_params:.4%}')

Total parameters: 66,955,010
Classification head parameters: 592,130
Fraction trainable if backbone is frozen: 0.8844%


In [22]:
#fraction that get updates:[('pre_classifier'+'classifier')/total params]*100
print((592130/66955010)*100)

0.8843699672362083


### Task 2 answer -- (b) and (c)
(b) Based on these two counts, what fraction of the model's parameters will actually receive gradient updates if you freeze the backbone?
(c) In 1-2 sentences, predict how this should affect training speed and memory use compared to fine-tuning the whole model -- you will check this prediction in Task 5.

_Your fraction-trainable interpretation (b) and your speed/memory prediction (c) here._

(b)If the backbone is frozen, only classification head (pre_classifier, classifier) gets updated (the backbone wont get gradient updates). Based on the results, about 0.88% of the models parameters receive gradient updates so we could say we are pre-training the model via fine-tuning and a very small number of parameters are being updated during training so it can adapt to learning our specific task.

(c)In this task, 0.88% of params are trainable, so about 99% is being frozen. if we wanted to finetune the whole model, all the model would have been finetuned and we would have updated every parameter (100%). So, training speed is slower if we want to do full fine-tunning compared to fine tunning only 0.88% of parameters. Also, updating only 0.88% of the parameters uses much less memory as most of the model stays frozen compared to when we are when we fine tune the whole model and 100% of parameters (uses much more memory).

## Task 3 (5 points): Grid search with a frozen backbone

Implement grid search for at least one hyperparameter (e.g., learning rate). For **each** setting:
1. Load a **freshly** pretrained model (do not reuse a model that already trained under a previous setting -- see the warning in the assignment).
2. Freeze every parameter in `model.distilbert` (the backbone) by setting `requires_grad=False`, so that only the classification head you identified in Task 2 will train.
3. Build an optimizer over *only* the head's trainable parameters (`filter(lambda p: p.requires_grad, model.parameters())` is one way to do this).
4. Train with `train_for_epochs(model, train_loader, val_loader, optimizer, GRID_EPOCHS)` and record the final validation F1-macro.

Report the F1-macro for every setting you tried and identify the best one.

In [32]:
# BEGIN YOUR CODE HERE (~12-18 lines)
#learning_rates = [0.00001,0.0001,0.001, 0.01, 0.1]  # try at least 3 values
learning_rates = [0.001, 0.01,1e-4, 2e-5, 3e-5, 5e-5] 
grid_search_results = {}

for lr in learning_rates:
    print(f'--- learning rate = {lr} ---')
    # 1. Load a fresh pretrained model. Remember attn_implementation=ATTN_IMPL
    #    (see the Setup cell) -- needed for training to work at all on MPS.
    newmodel= DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, attn_implementation=ATTN_IMPL)
    newmodel.to(device)

    # 2. Freeze every backbone parameter.
    #Freeze every parameter in `model.distilbert` (the backbone) by setting `requires_grad=False`, 
    # so that only the classification head you identified in Task 2 will train.
    for param in newmodel.distilbert.parameters():
        param.requires_grad = False

    # 3. Build an optimizer over only the trainable (head) parameters.
    #Build an optimizer over *only* the head's trainable parameters (`filter(lambda p: p.requires_grad, model.parameters())` 
    # is one way to do this).
    optimizer=optim.AdamW(filter(lambda p: p.requires_grad, newmodel.parameters()), lr=lr)

    # 4. Train, then record this setting's final validation F1-macro in
    #    grid_search_results[lr].
    #Train with `train_for_epochs(model, train_loader, val_loader, optimizer, GRID_EPOCHS)` 
    # and record the final validation F1-macro.
    newmodel=train_for_epochs(newmodel, train_loader, val_loader, optimizer, GRID_EPOCHS) #grid_epoch=2 
    valf1,val_preds,val_labels=evaluate(newmodel, val_loader)
    #getting final val f1-macro
    grid_search_results[lr]={'valf1': valf1, 'val_preds': val_preds, 'val_labels': val_labels}
   

# END YOUR CODE HERE

best_lr = max(grid_search_results, key= lambda lr: grid_search_results[lr]['valf1'])
print(
    f"\nBest learning rate: {best_lr} "
    f"(val F1-macro = {grid_search_results[best_lr]['valf1']:.4f})"
)

--- learning rate = 0.001 ---


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1/2 -- train loss: 0.3509, val F1-macro: 0.4929
  epoch 2/2 -- train loss: 0.3295, val F1-macro: 0.5553
--- learning rate = 0.01 ---


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1/2 -- train loss: 0.3946, val F1-macro: 0.4663
  epoch 2/2 -- train loss: 0.3578, val F1-macro: 0.4663
--- learning rate = 5e-05 ---


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1/2 -- train loss: 0.3819, val F1-macro: 0.4663
  epoch 2/2 -- train loss: 0.3513, val F1-macro: 0.4663

Best learning rate: 0.01 (val F1-macro = 0.5683)


In [24]:
print(type(valf1))
print(grid_search_results)

<class 'numpy.float64'>
{0.001: 0.5375207013385599, 0.01: 0.4663101604278075, 0.0001: 0.5156144784825403, 2e-05: 0.4663101604278075, 3e-05: 0.4663101604278075, 5e-05: 0.4717396200915985}


## Task 4 (3 points): Final frozen-head training

Using `best_lr` from Task 3, load one more fresh pretrained model, freeze the backbone the same way, and train on **train+val combined** (`trainval_loader`, given above) for `FINAL_EPOCHS` epochs. Then evaluate its F1-macro on the **test** set -- this is the number you'll compare against Task 5. Save the model as `frozen_head_model`, its test F1-macro as `frozen_head_f1`, and the wall-clock training time (via `time.time()`) as `frozen_head_train_time`; you'll need all three later.

In [53]:
# BEGIN YOUR CODE HERE (~8 lines)
# Load a fresh pretrained model (remember attn_implementation=ATTN_IMPL,
# see the Setup cell),
newmodel2= DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, attn_implementation=ATTN_IMPL)
newmodel2.to(device)

#  freeze the backbone, 
for param in newmodel2.distilbert.parameters():
    param.requires_grad = False

# build an optimizer over the head parameters using best_lr, 
optimizer2=optim.AdamW(filter(lambda p: p.requires_grad, newmodel2.parameters()), lr=best_lr)

#start time recording start
start_time = time.time()
# then train on trainval_loader for FINAL_EPOCHS epochs. 
newmodel2=train_for_epochs(newmodel2, trainval_loader, val_loader, optimizer2, FINAL_EPOCHS)


#Then evaluate its F1-macro on the **test** set -- this is the number you'll compare against Task 5.
#valf1,_,_=evaluate(newmodel2, test_loader)

# Record start_time and end_time around the
# training call using time.time() (the given line below uses them to
# compute the elapsed training time), and save your trained model as
# frozen_head_model.
#start_time = time.time()

#end time recording
end_time = time.time()


#Save the model as `frozen_head_model`, its test F1-macro as `frozen_head_f1`, 
# and the wall-clock training time (via `time.time()`) as `frozen_head_train_time`
frozen_head_model=newmodel2

# END YOUR CODE HERE

frozen_head_train_time = end_time - start_time
frozen_head_f1, frozen_head_preds, frozen_head_labels = evaluate(frozen_head_model, test_loader)

print(f'Frozen-head model: test F1-macro = {frozen_head_f1:.4f}, training time = {frozen_head_train_time:.1f}s')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1/3 -- train loss: 0.3737, val F1-macro: 0.4929
  epoch 2/3 -- train loss: 0.3377, val F1-macro: 0.4877
  epoch 3/3 -- train loss: 0.3244, val F1-macro: 0.5939
Frozen-head model: test F1-macro = 0.6091, training time = 137.8s


In [54]:
# Optional: training the Task 5 model on top of an already-loaded
# frozen_head_model can push you over your GPU's memory limit (e.g. a
# CUDA/MPS out-of-memory error, or your Colab runtime crashing). If that
# happens, uncomment the lines below to free frozen_head_model's memory
# before continuing -- frozen_head_f1 and frozen_head_train_time (both
# already saved above) are all Task 5 actually needs. Only do this if
# you're fine re-training frozen_head_model later, in case it turns out
# to be the better model for Task 6.

# import gc
# del frozen_head_model
# gc.collect()
# if device.type == 'cuda':
#     torch.cuda.empty_cache()
# elif device.type == 'mps':
#     torch.mps.empty_cache()

## Task 5 (3 points): Is freezing the backbone actually worth it?

Repeat the final training from Task 4, but this time **unfreeze every parameter** (leave `requires_grad=True` everywhere -- this is the default for a freshly loaded model, so simply skip the freezing step) and fine-tune the whole model, using a smaller learning rate suitable for full fine-tuning (e.g. `2e-5`). Train on `trainval_loader` for `FINAL_EPOCHS` epochs, exactly as in Task 4, and time it the same way. Save the model as `full_finetune_model`, its test F1-macro as `full_finetune_f1`, and its training time as `full_finetune_train_time`.

In [55]:
# BEGIN YOUR CODE HERE (~6 lines)
# Load a fresh pretrained model (remember attn_implementation=ATTN_IMPL,
# see the Setup cell; do NOT freeze anything this time), build
# an optimizer over ALL of its parameters with a small learning rate
# (e.g. 2e-5), then train on trainval_loader for FINAL_EPOCHS epochs.
# Record start_time and end_time using time.time() the same way as
# Task 4, and save your trained model as full_finetune_model.
newmodel3= DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, attn_implementation=ATTN_IMPL)
newmodel3.to(device)

lr3 = 2e-5
optimizer3=optim.AdamW(newmodel3.parameters(), lr=lr3)

start_time = time.time()
newmodel3=train_for_epochs(newmodel3, trainval_loader, val_loader, optimizer3, FINAL_EPOCHS)
end_time = time.time()

full_finetune_model=newmodel3


# END YOUR CODE HERE

full_finetune_train_time = end_time - start_time
full_finetune_f1, full_finetune_preds, full_finetune_labels = evaluate(full_finetune_model, test_loader)

print(f'Full fine-tune model: test F1-macro = {full_finetune_f1:.4f}, training time = {full_finetune_train_time:.1f}s')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1/3 -- train loss: 0.3353, val F1-macro: 0.4717
  epoch 2/3 -- train loss: 0.2706, val F1-macro: 0.7645
  epoch 3/3 -- train loss: 0.2105, val F1-macro: 0.9287
Full fine-tune model: test F1-macro = 0.7006, training time = 366.1s


### Task 5 answer

Report: (a) `frozen_head_f1` vs. `full_finetune_f1`, and (b) `frozen_head_train_time` vs. `full_finetune_train_time` (and how that ratio compares to your Task 2(b)/(c) prediction). In 2-3 sentences, discuss whether the accuracy gained (if any) was worth the extra compute, and connect this to what freezing the backbone actually does to the optimization problem.

_Your answer here._
Below is a comparison of the performance of training only the classifier head vs finetuning the entire model
| | F1_macro | Training Time, secs |
|--|--|--|
|frozen head model | 0.61| 137|
|fully fine tuned model| 0.70 | 366|

The fully fine tuned model takes approximately twice as long to train compared to the frozen model. This difference in training time is expected as we are updating significantly more weights in the fully fine tuned model.

With that said, the fully fine tuned model achieved an F1 macro of 0.7 vs 0.61 from the frozen model, an approximately 15% improvement. Considering the amount of compute resources needed to achieve a 15% improvement, it may not be worth it to fine-tune the entire model - it may be better to train the frozen model for longer or to fine-tune a subset of the layers in the backbone model. 

Freeing an already trained backbone limits the number of trainable parameters thus, reducing the training time and the compute resources needed. Furthermore since the frozen backbone has already been pretrained, those frozen weights can do a good job at extracting patterns from the input data, while focusing on training the classifier head will make the model more specialized to the current problem.



## Task 6 (3 points): Interpreting F1-macro

While running your Task 3 grid search, you may have noticed that some hyperparameter settings cause the model to always predict the same class (e.g., always predicting the negative class) regardless of input -- a common failure mode on this imbalanced dataset. Identify such a setting among your grid-search results if one occurred (if none did, pick your worst-performing setting instead), and report its F1-macro. Compute by hand what the precision and recall for *each* class must be for a model that always predicts a single label, and use that to explain why F1-macro is low in this case even though overall accuracy can look reasonable. In 2-3 sentences, discuss why F1-macro (rather than accuracy) is the right metric to watch on this dataset.

In [56]:
# BEGIN YOUR CODE HERE
# Identify the (setting, F1-macro) pair from grid_search_results that
# always-predicts-one-class (or your worst setting, if none collapsed),
# and print it here to support your written answer below.
worst_lr = min(grid_search_results, key= lambda lr: grid_search_results[lr]['valf1'])
print(
    f"Worst learning rate: {worst_lr} "
    f"(val F1-macro = {grid_search_results[worst_lr]['valf1']:.4f})"
)

# END YOUR CODE HERE

Worst learning rate: 2e-05 (val F1-macro = 0.4663)


In [57]:
worst_preds = grid_search_results[worst_lr]['val_preds']
worst_labels = grid_search_results[worst_lr]['val_labels']

In [58]:
np.all(np.array(worst_preds) == worst_preds[0]) # predicts the same class for every example in the validation set

True

In [59]:
labels_0 = np.sum(np.array(worst_labels) == 0)
labels_1 = np.sum(np.array(worst_labels) == 1)
labels_0, labels_1

(189, 1308)

In [60]:
## computes the TP TN FP FN for prcision and recall by class

#treat class 1 as positive and class 0 as negative
TP_class1 = sum((np.array(worst_preds) == 1) & (np.array(worst_labels) == 1))
TN_class1 = sum((np.array(worst_preds) == 0) & (np.array(worst_labels) == 0))
FP_class1 = sum((np.array(worst_preds) == 1) & (np.array(worst_labels) == 0))
FN_class1 = sum((np.array(worst_preds) == 0) & (np.array(worst_labels) == 1))

# treat class 0 as positive and class 1 as negative
TP_class0 = sum((np.array(worst_preds) == 0) & (np.array(worst_labels) == 0))
TN_class0 = sum((np.array(worst_preds) == 1) & (np.array(worst_labels) == 1))
FP_class0 = sum((np.array(worst_preds) == 0) & (np.array(worst_labels) == 1))
FN_class0 = sum((np.array(worst_preds) == 1) & (np.array(worst_labels) == 0))

In [61]:
# treat class 1 as positive and class 0 as negative
TP_class1, TN_class1, FP_class1, FN_class1

(1308, 0, 189, 0)

In [62]:
# treat class 0 as positive and class 1 as negative
TP_class0, TN_class0, FP_class0, FN_class0

(0, 1308, 0, 189)

### Task 6 answer

_Your by-hand precision/recall computation and your F1-macro-vs-accuracy discussion here._

Here is a summary of what the Precision and Recall were for the worst model mentioned above, which also predicts the same label:
| | Precision | Recall |
|--|--|--|
| Class 1| 0.87| 1|
|Class 0 | 0|0|

These numbers lead to a macro F1 score of 0.46.

As can be seen, the Precision and Recall scores for class 1 are high, but are 0 for class 0. This is indicative of a model that only predicts class 1. This leads to a low F1-macro score because that metric measures an unweighted average of the Precision and Recall for both classes. Since class 0 has a Precisiona and Recall of 0, those low scores will severely impact the F1-macro score. 

Condisering that the dataset is imbalanced and a model only predicts the majority class, there is a good chance that the accuracy of the model will be reasonable as the model will accurately predict most labels, however, such a model may never predict the minority class.

The F1-macro score is the appropriate metric to watch for this dataset as it is imbalanced and we want accurately predict both classes. Using this metric allows us to penalize models that have a low Precison and/or Recall score that may have a high accuracy.


## Bonus (2 points): Push F1-macro above 0.50

Apply a principled change to your code in order to achieve F1-macro $>0.50$ (recall the dataset is imbalanced -- a model that always predicts the majority class gets high accuracy but poor F1-macro). Explain what you did and why you did it.

In [63]:
# BEGIN YOUR CODE HERE
# Apply your principled change (e.g. a class-weighted loss, a different
# training schedule, more epochs, a different learning rate, etc.),
# retrain, and report the resulting test F1-macro. If you load a fresh
# model here too, remember attn_implementation=ATTN_IMPL (see the Setup
# cell).

# change the loss criteron to a weighted loss to address class imbalance
new_criterion = torch.nn.CrossEntropyLoss(weight=torch.tensor([labels_1, labels_0], dtype=torch.float).to(device))

new_model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, attn_implementation=ATTN_IMPL)
new_model.to(device)

# freeze the backbone parameters
for param in new_model.distilbert.parameters():
    param.requires_grad = False

# optimize the worst learning rate from the grid search
new_optimizer = optim.AdamW(filter(lambda p: p.requires_grad, new_model.parameters()), lr=worst_lr)

# define a new training function that uses the weighted loss
def train_one_epoch_weighted(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = criterion(outputs.logits, batch['labels'])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def train_for_epochs_weighted(model, train_loader, val_loader, optimizer, criterion, num_epochs):
    for epoch in range(num_epochs):
        train_loss = train_one_epoch_weighted(model, train_loader, optimizer, criterion)
        val_f1, _, _ = evaluate(model, val_loader)
        print(f'  epoch {epoch + 1}/{num_epochs} -- train loss: {train_loss:.4f}, val F1-macro: {val_f1:.4f}')
    return model

new_model = train_for_epochs_weighted(new_model, train_loader, val_loader, new_optimizer, new_criterion, GRID_EPOCHS)

new_model_f1_macro, new_model_preds, new_model_labels = evaluate(new_model, test_loader)

print(f'New model with weighted loss: test F1-macro = {new_model_f1_macro:.4f}')



# END YOUR CODE HERE

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1/2 -- train loss: 0.6787, val F1-macro: 0.6551
  epoch 2/2 -- train loss: 0.6516, val F1-macro: 0.6309
New model with weighted loss: test F1-macro = 0.6122


### Bonus answer

_What you changed and why._

To address the bonus problem, the worst performing model from the grid search was improved. That model was the one in which the learning rate was 2e-05, it only predicted the majority class and it achieved an F1-macro score of 0.46.

To improved the model, the original loss criteron was changed to a weighted loss. ` torch.nn.CrossEntropyLoss()` was used and a weight tensor was passed as an argument. This weight tensor was a 1-D tensor of shape `(number of classes, 1)` and was used to scale the minority class by a larger number and the majority class by a smaller number. Doing this helps to train the model to pay more attention to the minority class.

With the addition of a new criteron, the `train_per_epoch()` and `train_for_epoch()` functions were updated, the model was trained and achieved an F1-macro score of 0.61.